In [ ]:
from fmsae_repro import MODELS, RESULTS, load_manifest
import os
import joblib
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd
import zarr
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
NEW_CLUSTERERS_DIR = f"{MODELS}/token_clusterers/ablation"

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)

In [ ]:
feature_table = pd.read_csv(f'{RESULTS}/tables/Supplementary_Table_3.csv')
selected_groups = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,27]
feature_table = feature_table[feature_table['final_group_id'].isin(selected_groups)]
feature_ids = list(feature_table['Feature ID'].to_list()) # these are all non-blank, non-necrotic features

In [ ]:
from mesoslide.tools._model_stage import _resolve_model

# Resolved through mesoslide so `uni` carries the same transform used for the stored
# embeddings (bilinear resize per UNI's timm config), not lazyslide_models' bicubic default.
uni, _ = _resolve_model("uni")

First retrain all token clusterers, as we found a bug in the selection of top patches...

In [ ]:
# Ablation ordering: clusters ranked by how much removing their tokens lowers the SAE
# feature score, beyond removing the same number of random tokens.
ABLATION_CLUSTERERS_DIR = f"{MODELS}/token_clusterers/ablation"
os.makedirs(ABLATION_CLUSTERERS_DIR, exist_ok=True)

sae = joblib.load(f"{MODELS}/ref_SAE.joblib")

for fid in tqdm(feature_ids):
    out_path = f"{ABLATION_CLUSTERERS_DIR}/UNI_SAE_{fid}.pkl"
    if os.path.exists(out_path):
        continue
    c = ms.tools.fit_token_clusterer(slides, f"UNI_SAE_{fid}", model=uni,
                            clusterer=ms.tools.TokenClusterer(n_clusters=3, 
                                                              ordering="ablation", 
                                                              random_state=0,
                                                              n_init=10),
                            scorer=sae,
                            batch_size=128,
                            n_patches=100, 
                            top_fraction=0.1, 
                            device="cuda:2",
                            progress_bar=False)
    joblib.dump(c, out_path)

Now get top patches for each feature and cache the dense token embeddings:

In [ ]:
from mesoslide._patch_data import *

all_patches = {}
for feature_id in tqdm(feature_ids):
    output_dir = f"{RESULTS}/SAE_feature_high_scoring_patches/UNI_SAE_{feature_id}.zarr"
    if os.path.exists(output_dir):
        patches = read_patch_data(output_dir)
        all_patches[feature_id] = patches
        continue
    clusterer_path = os.path.join(NEW_CLUSTERERS_DIR, f"UNI_SAE_{feature_id}.pkl")
    clusterer = joblib.load(clusterer_path)

    patches = ms.select_top_patches(slides, f"UNI_SAE_{feature_id}", n=100, top_fraction=0.1)
    ms.preprocessing.extract_cluster_maps(patches, clusterer, model=uni, device='cuda:0')
    patches.write(output_dir, overwrite=True)
    zarr.consolidate_metadata(output_dir)
    all_patches[feature_id] = patches

load all clusterers

In [ ]:
all_clusterers = {}
for feature_id in tqdm(feature_ids):
    clusterer_path = os.path.join(NEW_CLUSTERERS_DIR, f"UNI_SAE_{feature_id}.pkl")
    all_clusterers[feature_id] = joblib.load(clusterer_path)

For each SAE feature, for its top scoring patches, use all clusterers to produce token-level cluster labels (14×14 per patch). Computed once from the cached `UNI_dense` embeddings and saved to `token_labels.npy`; delete that file after retraining clusterers so it is recomputed.

Token-level IoU equals the pixel-level IoU of the 448×448 cluster maps, since each token maps to an exact 32×32 pixel block.

In [ ]:
SIM_DIR = f"{RESULTS}/SAE_feature_high_scoring_patches"
LABELS_PATH = os.path.join(SIM_DIR, "token_labels.npy")
clusterer_list = [all_clusterers[fid] for fid in feature_ids]

if os.path.exists(LABELS_PATH):
    token_labels = np.load(LABELS_PATH)
else:
    per_feature = []
    for fid in tqdm(feature_ids):
        # Token embeddings of this feature's 100 patches, loaded above
        dense = all_patches[fid].tables["tiles_table"].obsm["UNI_dense"]   # (100, 196, 1024)
        # Labels from every clusterer on these patches: (n_clusterers, 100, 14, 14)
        per_feature.append(ms.tools.predict_token_labels(clusterer_list, dense))
    token_labels = np.stack(per_feature)   # (n_features, n_clusterers, 100, 14, 14)
    np.save(LABELS_PATH, token_labels)

token_labels.shape

In [ ]:
def compute_iou_1d(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Compute intersection (both are True) and union (either is True)
    intersection = np.logical_and(arr1, arr2).sum()
    union = np.logical_or(arr1, arr2).sum()
    
    # Handle division by zero if both arrays are entirely False
    if union == 0:
        return 0.0
        
    return float(intersection / union)

index_of = {fid: k for k, fid in enumerate(feature_ids)}

def compute_micro_iou(feature_id1, feature_id2):
    """IoU of the top cluster of clusterer 1 vs. clusterer 2, on feature 1's patches."""
    i, j = index_of[feature_id1], index_of[feature_id2]
    top_1 = all_clusterers[feature_id1].n_clusters_ - 1   # 2 when n_clusters=3
    top_2 = all_clusterers[feature_id2].n_clusters_ - 1
    fg_1 = token_labels[i, i].ravel() == top_1   # clusterer 1 on its own patches
    fg_2 = token_labels[i, j].ravel() == top_2   # clusterer 2 on feature 1's patches
    return compute_iou_1d(fg_1, fg_2)

def compute_micro_iou_best(feature_id1, feature_id2, return_cluster=False):
    """Highest IoU between clusterer 1's top cluster and any cluster (0..top_2) of
    clusterer 2, on feature 1's patches. With return_cluster=True, also returns the
    matching cluster id of clusterer 2."""
    i, j = index_of[feature_id1], index_of[feature_id2]
    top_1 = all_clusterers[feature_id1].n_clusters_ - 1
    top_2 = all_clusterers[feature_id2].n_clusters_ - 1
    fg_1 = token_labels[i, i].ravel() == top_1
    labels_2 = token_labels[i, j].ravel()
    ious = [compute_iou_1d(fg_1, labels_2 == k) for k in range(top_2 + 1)]
    best = int(np.argmax(ious))
    return (ious[best], best) if return_cluster else ious[best]

In [ ]:
# iou_matrix[i, j]: clusterer i vs. clusterer j on feature i's patches (not symmetric)
iou_matrix = pd.DataFrame(
    [[compute_micro_iou(a, b) for b in feature_ids] for a in feature_ids],
    index=feature_ids, columns=feature_ids,
)
iou_matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 20))
first_n =100
sns.heatmap(iou_matrix.iloc[:first_n, :first_n], cmap="magma", 
            square=True, cbar_kws={"label": "IoU"}, 
            xticklabels=feature_ids[:first_n], 
            yticklabels=feature_ids[:first_n])

### Partition similarity (ARI)

IoU compares only the salient (top) cluster of each clusterer. The adjusted Rand index compares the whole 3-cluster partition of the same tokens and ignores cluster labels. Two features that divide the tissue the same way score high even if a different cluster is salient for each, e.g. compositional features where either of two clusters can be salient depending on the feature direction. ARI is 1 for identical partitions, about 0 for chance agreement, and can be negative.

Like IoU, `ari_matrix[i, j]` is computed on feature i's patches, so it is not symmetric.

In [ ]:
from sklearn.metrics import adjusted_rand_score


def compute_micro_ari(feature_id1, feature_id2):
    """ARI between clusterer 1's and clusterer 2's partitions of feature 1's patch tokens."""
    i, j = index_of[feature_id1], index_of[feature_id2]
    return adjusted_rand_score(token_labels[i, i].ravel(), token_labels[i, j].ravel())


# ari_matrix[i, j]: clusterer i vs. clusterer j on feature i's patches (not symmetric)
ari_matrix = pd.DataFrame(
    [[compute_micro_ari(a, b) for b in feature_ids] for a in feature_ids],
    index=feature_ids, columns=feature_ids,
)
ari_matrix

In [ ]:
plt.figure(figsize=(20, 20))
first_n = 100
sns.heatmap(ari_matrix.iloc[:first_n, :first_n], cmap="magma", vmin=0, vmax=1,
            square=True, cbar_kws={"label": "ARI"},
            xticklabels=feature_ids[:first_n],
            yticklabels=feature_ids[:first_n])

### Best-match IoU

For each pair, feature 1's salient tokens are compared with every cluster of clusterer 2, and the highest IoU is kept. It is at least the plain IoU. `iou_best_cluster[i, j]` gives clusterer j's cluster that matched best: 2 (its salient cluster) means the two features agree on the salient region; 0 or 1 means feature i's salient region corresponds to a non-salient cluster of feature j (e.g. compositional features).

In [ ]:
# iou_best_matrix[i, j]: best IoU of clusterer i's salient cluster vs. any cluster of
# clusterer j, on feature i's patches (not symmetric); iou_best_cluster[i, j]: that cluster
best_pairs = [[compute_micro_iou_best(a, b, return_cluster=True) for b in feature_ids] for a in feature_ids]
arr = np.array([[iou for iou, _ in row] for row in best_pairs])
arr_symmetrized = np.minimum(arr, arr.T)  # symmetrize by taking the minimum of iou(i,j) and iou(j,i)
iou_best_matrix = pd.DataFrame(arr_symmetrized,
                               index=feature_ids, 
                               columns=feature_ids)
iou_best_cluster = pd.DataFrame([[k for _, k in row] for row in best_pairs],
                                index=feature_ids, 
                                columns=feature_ids)
iou_best_matrix

In [ ]:
dist = 1.0 - iou_best_matrix.to_numpy()
np.fill_diagonal(dist, 0.0)

g, linkage_matrix, order, clusters, _ = ms.plotting.plot_clustered_heatmap(
    dist,                                   # cluster on distance
    display_matrix=iou_best_matrix.to_numpy(),  # show similarity
    is_distance_matrix=True,
    linkage_method="average",
    cmap="Blues",
    threshold=25, 
    criterion='maxclust'
)
ax = g.ax_heatmap
positions = np.arange(len(ordered_ids)) + 0.5          # cell centres
ax.set_xticks(positions)
ax.set_yticks(positions)
ax.set_xticklabels(ordered_ids, rotation=90, fontsize=6)
ax.set_yticklabels(ordered_ids, rotation=0, fontsize=6)
ax.tick_params(bottom=True, labelbottom=True, right=True, labelright=True)
g.fig.set_size_inches(20, 20)                           
ordered_ids = [feature_ids[k] for k in order]   # feature ids in dendrogram order
feature_cluster = pd.Series(clusters, index=ordered_ids, name="cluster")


In [ ]:
test_clusterers = {}

In [ ]:
fid = 13679
c = ms.tools.segmenters.fit_token_clusterer(slides, f"UNI_SAE_{fid}", model=uni,
                        clusterer=ms.tools.TokenClusterer(n_clusters=3, random_state=0),
                        batch_size=100, top_fraction=1.0, min_score=0.0,
                        n_patches=100)
test_clusterers[fid] = c

In [ ]:
test_clusterers[13679].plot_cluster_feature_correlation()

In [ ]:
test_patches = ms.select_top_patches(slides, f'UNI_SAE_{6261}', 
                                     top_fraction=0.1, n=20)


In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    test_patches, [all_clusterers[6261], all_clusterers[18376], all_clusterers[10512], 
                   all_clusterers[41683]],
    return_fig=True,
    cmap='viridis',
    model=uni
)

In [ ]:
import matplotlib.pyplot as plt

def plot_cluster_ablation(c, axes=None):
    """Per-patch ablation diagnostics of a clusterer fit with ordering='ablation'."""
    res, summary = c.ablation_results_, c.ablation_summary_
    colors = plt.cm.viridis(np.linspace(0, 1, c.n_clusters_))
    if axes is None:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    else:
        fig = axes[0].figure
    top = res[["score_label_removed", "score_random_removed"]].to_numpy().max() * 1.05
    for k in summary.index:
        r, s = res[res["cluster"] == k], summary.loc[k]
        axes[0].scatter(r["score_random_removed"], r["score_label_removed"], s=16, alpha=0.7, color=colors[k],
                        label=f"Cluster {k}: excess drop {s['excess_drop']:.3f} (p={s['wilcoxon_p']:.2g}, n={int(s['n_patches'])})")
        axes[1].scatter(r["token_fraction"], r["excess_drop"], s=16, alpha=0.7, color=colors[k])
    axes[0].plot([0, top], [0, top], "k--", lw=1)
    axes[0].set(xlabel="Score, random tokens removed", ylabel="Score, cluster tokens removed",
                xlim=(0, top), ylim=(0, top))
    axes[0].legend(loc="upper left", fontsize=8, frameon=False)
    axes[1].axhline(0, color="k", lw=1, ls="--")
    axes[1].set(xlabel="Fraction of patch tokens in cluster", ylabel="Excess drop")
    fig.suptitle(f"{c.display_name}: token ablation (below diagonal = cluster matters beyond its size)")
    fig.tight_layout()
    return fig, axes

c = joblib.load(os.path.join(ABLATION_CLUSTERERS_DIR, "UNI_SAE_43675.pkl"))
_ = plot_cluster_ablation(c)


In [ ]:
patches = ms.select_top_patches(slides, 'UNI_SAE_41985', top_fraction=1.0, n=50000)
patches.tables['tiles_table'].obs['_feature_score'].hist(bins=50)

In [ ]:
np.percentile(patches.tables['tiles_table'].obs['_feature_score'], 90)

### Token ablation: which cluster does the SAE feature depend on?

The SAE reads UNI's patch-level (CLS) embedding, not individual tokens, so a cluster's contribution is measured by removing its tokens and recomputing the feature score:

1. For each patch and each cluster, re-run UNI with that cluster's tokens removed. They are masked out as attention keys in every layer, which gives the same CLS output as running on the kept tokens only, with no pixels painted over.
2. Read the feature score from the new CLS embedding with the SAE encoder.
3. Control: remove the same number of randomly chosen tokens. `excess_drop` = drop from removing the cluster's tokens − drop from removing random tokens. It is positive when the cluster matters more than its size alone would explain.

Set `DEVICE` in the setup cell to a free GPU (each run cell takes ~10 s on an L40S; ~15 min on CPU).

In [ ]:
import torch
from scipy.stats import wilcoxon

# SAE feature activation from a patch-level UNI embedding x: relu(w_f . (x * scale) + b_f)
sae = joblib.load(f"{MODELS}/ref_SAE.joblib")
SAE_W = sae.model_.encoder.weight.detach().double()
SAE_B = sae.model_.encoder.bias.detach().double()
SAE_SCALE = float(sae.scale_factor_)


def sae_feature_scores(cls_embeddings, feature_ids):
    """SAE activations, shape (n_embeddings, n_features), of the given feature ids."""
    idx = torch.as_tensor(feature_ids)
    return torch.relu(cls_embeddings.double() * SAE_SCALE @ SAE_W[idx].T + SAE_B[idx])


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"   # set to a free GPU
uni.to(DEVICE)
uni.model.eval()
uni_transform = uni.get_transform()   # corrected transform, since uni came from _resolve_model


@torch.inference_mode()
def cls_with_kept_tokens(image, keep_masks):
    """UNI patch embedding (CLS) of one patch under several token-keep masks.

    image: (3, H, W) uint8. keep_masks: (n_variants, n_tokens) bool, True = keep.
    Removed tokens are excluded as attention keys in every layer, so the CLS output
    equals running UNI on the kept tokens alone.
    """
    x = uni_transform(torch.as_tensor(np.asarray(image)).unsqueeze(0)).to(DEVICE)   # uint8 in
    x = x.expand(len(keep_masks), -1, -1, -1)
    prefix = torch.ones(len(keep_masks), uni.model.num_prefix_tokens, dtype=torch.bool)
    key_mask = torch.cat([prefix, torch.as_tensor(keep_masks)], dim=1)[:, None, None, :].to(DEVICE)
    features = uni.model.forward_features(x, attn_mask=key_mask)
    return uni.model.forward_head(features).cpu()

In [ ]:
# Sanity check: with no tokens removed, the recomputed embedding and SAE score should
# match what is stored for these patches.
check_table = all_patches[41985].tables["tiles_table"]
n_tokens = check_table.obsm["UNI_dense"].shape[1]
full = torch.cat([
    cls_with_kept_tokens(check_table.obsm["he_patch_img"][p], np.ones((1, n_tokens), bool))
    for p in range(3)
])
stored = torch.as_tensor(np.asarray(check_table.obsm["UNI_embedding"][:3]))
print("max |embedding difference|:", (full - stored).abs().max().item())
print("recomputed scores:", sae_feature_scores(full, [41985]).ravel().numpy().round(4))
print("stored scores:    ", check_table.obs["_feature_score"].to_numpy()[:3].round(4))

In [ ]:
def cluster_ablation(patches, clusterers, feature_ids, seed=0):
    """Feature score drop when each cluster's tokens are removed from each patch.

    For every patch, clusterer and cluster present in the patch, removes that
    cluster's tokens ("cluster_tokens") and, as a control, the same number of
    randomly chosen tokens ("random_tokens"), then recomputes the SAE scores of
    `feature_ids`. Returns one row per (patch, clusterer, cluster, removal, feature).
    """
    table = patches.tables["tiles_table"]
    images = table.obsm["he_patch_img"]   # (n_patches, 3, H, W) uint8
    dense = table.obsm["UNI_dense"]        # (n_patches, n_tokens, D)
    labels = {name: c.predict(dense).reshape(len(dense), -1) for name, c in clusterers.items()}
    rng = np.random.default_rng(seed)
    rows = []
    for p in tqdm(range(len(images))):
        keep_masks = [np.ones(dense.shape[1], bool)]   # first variant: nothing removed
        variants = []
        for name, patch_labels in labels.items():
            for k in range(clusterers[name].n_clusters_):
                removed = patch_labels[p] == k
                n_removed = int(removed.sum())
                if n_removed == 0:
                    continue
                random_removed = np.zeros_like(removed)
                random_removed[rng.choice(len(removed), n_removed, replace=False)] = True
                keep_masks += [~removed, ~random_removed]
                variants += [(name, k, "cluster_tokens", n_removed), (name, k, "random_tokens", n_removed)]
        scores = sae_feature_scores(cls_with_kept_tokens(images[p], np.stack(keep_masks)), feature_ids).numpy()
        for (name, k, removal, n_removed), variant_scores in zip(variants, scores[1:]):
            for j, fid in enumerate(feature_ids):
                rows.append(dict(
                    patch=p, clusterer=name, cluster=k, removal=removal,
                    token_fraction=n_removed / dense.shape[1], feature=fid,
                    full_score=scores[0, j], score=variant_scores[j],
                    drop=scores[0, j] - variant_scores[j],
                ))
    return pd.DataFrame(rows)


def summarize_ablation(results):
    """Per clusterer, cluster and feature: mean drop from removing the cluster's tokens,
    the random-token control, their difference (excess drop) and a Wilcoxon p-value."""
    key = ["clusterer", "cluster", "feature", "patch"]
    wide = results.pivot_table(index=key, columns="removal", values="drop").reset_index()
    wide = wide.merge(
        results.drop_duplicates(key)[key + ["token_fraction", "full_score"]], on=key,
    )
    wide["excess_drop"] = wide["cluster_tokens"] - wide["random_tokens"]
    summary = wide.groupby(["clusterer", "feature", "cluster"]).agg(
        n_patches=("patch", "size"),
        token_fraction=("token_fraction", "mean"),
        full_score=("full_score", "mean"),
        drop_cluster_tokens=("cluster_tokens", "mean"),
        drop_random_tokens=("random_tokens", "mean"),
        excess_drop=("excess_drop", "mean"),
    )
    summary["wilcoxon_p"] = wide.groupby(["clusterer", "feature", "cluster"])["excess_drop"].apply(
        lambda d: wilcoxon(d).pvalue if len(d) > 1 and np.any(d != 0) else np.nan
    )
    return summary.round(4)

In [ ]:
# 41985's top 100 patches. Clusterer 65169's cluster 2 is the RBC cluster, so
# including it measures the effect of removing RBC tokens directly.
ablation_clusterers = {
    "41985": all_clusterizers[41985],
    "65169": all_clusterizers[65169],
}
ablation_41985 = cluster_ablation(all_patches[41985], ablation_clusterers, feature_ids=[41985, 65169])
summarize_ablation(ablation_41985)

In [ ]:
# Positive control: on 65169's own top patches, removing its RBC cluster (2)
# should lower 65169 far more than removing random tokens.
ablation_65169 = cluster_ablation(all_patches[65169], ablation_clusterers, feature_ids=[65169, 41985])
summarize_ablation(ablation_65169)

In [ ]:
import importlib
import mesoslide.tools.segmenters._token_clusterer
import mesoslide._patch_selector
importlib.reload(mesoslide.tools.segmenters._token_clusterer)
importlib.reload(mesoslide._patch_selector)

In [ ]:
patches = ms.select_top_patches(slides, 'UNI_SAE_13679', top_fraction=1.0, n=10000)
patches.tables['tiles_table'].obs['_feature_score'].hist(bins=50)

In [ ]:
import mesoslide.plotting._gallery_plan
import mesoslide.plotting._patch_gallery
importlib.reload(mesoslide.plotting._gallery_plan)
importlib.reload(mesoslide.plotting._patch_gallery)

In [ ]:
_ = ms.plotting.plot_patch_gallery(patches, slides, show_scores=True, return_fig=True)

In [ ]:
importlib.reload(mesoslide.tools.segmenters.TokenClusterizer)

In [ ]:
from mesoslide.tools.segmenters.TokenClusterizer import TokenClusterizer
select_feature = 63030
clusterizer_1 = TokenClusterizer(
    model=uni,
    feature_name=f'UNI_SAE_{select_feature}_v1',
    device='cuda:0'
)
clusterizer_2 = TokenClusterizer(
    model=uni,
    feature_name=f'UNI_SAE_{select_feature}_v2',
    device='cuda:0'
)
# clusterizer_3 = TokenClusterizer(
#     model=uni,
#     feature_name='UNI_SAE_13679_v3',
#     device='cuda:0'
# )

In [ ]:
clusterizer_1.fit(slides, feature_name=f'UNI_SAE_{select_feature}', model=uni, top_fraction=0.10, n_positive=100, min_score=0.0)
clusterizer_1.plot_cluster_feature_correlation()

In [ ]:
clusterizer_2.fit(slides, feature_name=f'UNI_SAE_{select_feature}', model=uni, top_fraction=0.10, n_positive=500, min_score=0.0)
clusterizer_2.plot_cluster_feature_correlation()

In [ ]:
_ = ms.plotting.plot_patch_gallery_with_saliency(
    all_patches[63030], [all_clusterizers[63030], all_clusterizers[65162], 
                         clusterizer_1, clusterizer_2],
    return_fig=True,
    cmap='viridis',
    model=uni
)